# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing all data entities by their Croissant `@id` fields.

### Dataset Source
The dataset Croissant schema is available at:
<br>
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access high-level metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets and fields. All lookups are performed using the `@id` for each entity.

Let's enumerate all the available record sets (`@id` and name), and then show their available fields by `@id`.

In [ ]:
# List all record sets defined in the dataset, including their @id
print("Available Record Sets:")
record_sets = []
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set['@id']} : {record_set.get('name', '[No Name]')}")
    record_sets.append(record_set['@id'])

# For each record set, list the field @ids
print("\nFields by record set (@id):\n")
record_set_fields = {}
for record_set in dataset.metadata.record_sets:
    rs_id = record_set['@id']
    print(f"Record Set: {rs_id}")
    fields = record_set.get('fields', [])
    field_ids = []
    for field in fields:
        print(f"  - {field['@id']} : {field.get('name', '[No Name]')}")
        field_ids.append(field['@id'])
    record_set_fields[rs_id] = field_ids


## 3. Data Extraction
Extract data from one or more record sets using their `@id`. The resulting DataFrames will use the corresponding field `@id` as columns.

In [ ]:
# Prepare a list of record set @ids for extraction
print('Record Sets available:', record_sets)
dataframes = {}

for rs_id in record_sets:
    print(f"\nLoading records from record set '{rs_id}' ...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  DataFrame shape: {df.shape}")
        print(f"  Columns (field @id):")
        print(f"    {df.columns.tolist()}")
    else:
        print(f"  [No records found]")

# For further steps, select the first record set with data (if any)
chosen_record_set_id = None
for rs_id in record_sets:
    if rs_id in dataframes and not dataframes[rs_id].empty:
        chosen_record_set_id = rs_id
        break
if chosen_record_set_id is not None:
    df_main = dataframes[chosen_record_set_id]
    print(f"\nPreview of data from record set '{chosen_record_set_id}':\n")
    display(df_main.head())
else:
    print("\nNo record sets contained extractable tabular data.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing. We'll:
- Select a numeric field (by @id) for filtering and normalization
- Filter out rows based on a simple threshold
- Normalize a numeric column
- Optionally group by another field (e.g., a categorical field @id)

*(Adjust the field `@id`s below to match those found above!)*

In [ ]:
# If data was extracted, proceed:
if chosen_record_set_id is not None and not df_main.empty:
    print(f"Columns available for record set '{chosen_record_set_id}':")
    print(df_main.columns.tolist())

    # Attempt to locate a numeric field (e.g., a field @id containing 'log_likelihood' or 'coef')
    # You may need to manually change 'log_likelihood' to the correct @id found above
    numeric_candidates = [col for col in df_main.columns if 'log' in col.lower() or 'coef' in col.lower() or 'value' in col.lower()]
    if not numeric_candidates:
        numeric_candidates = df_main.select_dtypes(include=['int', 'float']).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"\nUsing numeric field: {numeric_field_id}")
    else:
        print("No obvious numeric column found. Please adjust 'numeric_field_id' manually.")

    # Example threshold
    threshold = 10
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        filtered_df = df_main[pd.to_numeric(df_main[numeric_field_id], errors='coerce') > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        mean = filtered_df[numeric_field_id].astype(float).mean()
        std = filtered_df[numeric_field_id].astype(float).std()
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id].astype(float) - mean) / std
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt to group by a categorical field
        # Try to pick a field with discrete or string values, e.g. 'variable' or similar
        group_candidates = [col for col in df_main.columns if col != numeric_field_id]
        group_field_id = None
        for col in group_candidates:
            if df_main[col].nunique() < 10:  # Arbitrary threshold: 10 groups
                group_field_id = col
                break
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
else:
    print("No main DataFrame to analyze.")

## 5. Visualization
Visualize distributions or relationships using the field `@id`. For example, a histogram of the numeric field, or a boxplot grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id is not None and not df_main.empty and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df_main[numeric_field_id].astype(float), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If a group_field_id exists, show a boxplot
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_main)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Load a dataset using its Croissant schema URL
- Enumerate record sets and fields by `@id`
- Extract tabular data for analysis referencing all entities by their `@id`
- Perform simple EDA: filtering, normalization, aggregation
- Visualize numeric fields with histograms and boxplots

**Next Steps:**
- Explore additional record sets or fields.
- Apply analytic or machine learning models relevant to your inquiry.
- Always document the use of the dataset, referencing field and set `@id` for reproducibility.
